<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png" width="250" style="display: block; margin-left: auto; margin-right: auto;">

# Titanic – Data Preparation and Feature Engineering

## Purpose

This notebook turns the main EDA findings into one clear and reproducible preprocessing workflow. The priority is to understand **what each step does and why it is necessary**.

By the end, we will be able to:

- validate the minimum input-data contract;
- create train and test partitions without fitting transformations beforehand;
- create a small set of features justified by the EDA;
- impute, encode, and scale different types of variables;
- assemble all transformations into a scikit-learn pipeline;
- prepare the handoff to model comparison without consulting the test set.

```text
Raw data
    ↓
Train / test
    ↓
Feature engineering
    ↓
Imputation + encoding + scaling
    ↓
Model
```

> An advanced companion notebook contains custom transformers, conditional age imputation, ticket-group features, configurable pipelines, and ablation experiments. The present notebook teaches the essential path first.

> **Test-set limitation:** the earlier EDA inspected the complete labeled dataset. Therefore, the test partition created here is useful for the course workflow, but it is not a perfectly untouched final test set. In a new project, create and lock the final test set before detailed EDA.

## 1. Imports and project paths

Dependencies should be installed from the project environment, not from a `pip install` cell inside the notebook.

In [1]:
from __future__ import annotations

import json
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Survived"

# Allow execution from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "titanic_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "titanic_data.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

Python: 3.12.3
pandas: 3.0.1
scikit-learn: 1.8.0


## 2. Load and validate the raw data

The EDA identified the expected columns and valid ranges. We now convert those observations into executable checks.

The validation is deliberately written with explicit steps. If the input schema changes, the notebook should stop with an informative message instead of silently ignoring the problem.

In [2]:
required_columns = [
    "PassengerId",
    "Survived",
    "Pclass",
    "Name",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked",
]

df = pd.read_csv(DATA_PATH)

missing_columns = []
for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if df["PassengerId"].duplicated().any():
    raise ValueError("PassengerId must be unique.")

if not df["Pclass"].isin([1, 2, 3]).all():
    raise ValueError("Pclass must contain only 1, 2, or 3.")

if df["Age"].dropna().lt(0).any():
    raise ValueError("Age cannot be negative.")

if df["Fare"].dropna().lt(0).any():
    raise ValueError("Fare cannot be negative.")

if not df[TARGET_COLUMN].isin([0, 1]).all():
    raise ValueError("Survived must contain only 0 or 1.")

print(f"Validated dataset: {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

Validated dataset: 891 rows and 12 columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Separate the target and create the partitions

We split before fitting imputers, encoders, scalers, or models. `stratify=y` keeps approximately the same target proportion in both partitions.

Only training data may be used for cross-validation, feature selection, model comparison, Optuna, or threshold selection. We deliberately avoid displaying the test-set target distribution.

In [3]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

training_target_summary = (
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)
training_target_summary

Training rows: 712
Test rows: 179


,proportion
Survived,
0,0.616573
1,0.383427


## 4. Translate EDA findings into preparation decisions

This notebook implements one understandable baseline. Alternative representations remain hypotheses for the modeling notebook.

| EDA finding | Decision in this baseline | Alternative to compare later |
|---|---|---|
| `Age` has structured missingness | Median imputation + missingness indicator | Conditional imputation by class/title |
| `Fare` is strongly right-skewed | Use `log1p(Fare)` | Keep the original fare |
| `Pclass` may not have a linear effect | One-hot encode it | Treat it as ordinal for tree models |
| Family structure is nonlinear | Use family size and traveling-alone indicator | Keep `SibSp` and `Parch` |
| Cabin availability may contain information | Use a cabin-availability indicator | Use grouped deck |
| Name contains social information | Extract and group passenger title | Exclude title |

The baseline avoids keeping exact redundant representations at the same time.

## 5. Create understandable features

The function below is **stateless**: it does not learn medians, category frequencies, or any information from other rows. It only combines values already present in each passenger record.

We first present it as an ordinary pandas function. Later, `FunctionTransformer` will place this same function inside the sklearn pipeline.

In [4]:
def create_titanic_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create deterministic features from the raw Titanic columns."""
    result = data.copy()

    # Extract the text between the comma and period in each passenger name.
    title_pattern = r",\s*([^.]+)\."
    result["Title"] = result["Name"].str.extract(
        title_pattern,
        expand=False,
    )

    # Normalize equivalent titles; rare titles will be grouped by the encoder.
    title_aliases = {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
    }
    result["Title"] = result["Title"].replace(title_aliases)
    result["Title"] = result["Title"].fillna("Unknown")

    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)
    result["HasCabin"] = result["Cabin"].notna().astype(int)
    result["AgeMissing"] = result["Age"].isna().astype(int)
    result["FareLog1p"] = np.log1p(result["Fare"])

    return result

### Inspect the result before building the pipeline

During development, inspect a few rows to verify that every new feature means what we intended. This preview is for teaching and debugging; the final transformation will still occur inside the pipeline.

In [5]:
preview = create_titanic_features(X_train.head())

columns_to_preview = [
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone",
    "HasCabin",
    "Age",
    "AgeMissing",
    "Fare",
    "FareLog1p",
    "Title",
]

preview[columns_to_preview]

,SibSp,Parch,FamilySize,IsAlone,HasCabin,Age,AgeMissing,Fare,FareLog1p,Title
692,0,0,1,1,0,NaN,1,56.4958,4.051712,Mr
481,0,0,1,1,0,NaN,1,0.0000,0.000000,Mr
527,0,0,1,1,1,NaN,1,221.7792,5.406181,Mr
855,0,1,2,0,0,18.0,0,9.3500,2.336987,Mrs
801,1,1,3,0,0,31.0,0,26.2500,3.305054,Mrs


## 6. Group features by the transformation they need

A `ColumnTransformer` lets us apply a different preparation strategy to each variable type.

| Group | Preparation |
|---|---|
| Numeric | Median imputation and standardization |
| Categorical | Most-frequent imputation and one-hot encoding |
| Binary indicators | Pass through unchanged |

Variables such as `PassengerId`, raw `Name`, `Cabin`, `Ticket`, `SibSp`, and `Parch` are not included in these lists, so `ColumnTransformer` will drop them.

In [6]:
numeric_features = [
    "Age",
    "FareLog1p",
    "FamilySize",
]

categorical_features = [
    "Sex",
    "Embarked",
    "Pclass",
    "Title",
]

binary_features = [
    "AgeMissing",
    "IsAlone",
    "HasCabin",
]

## 7. Build the numeric and categorical pipelines

### Numeric variables

1. `SimpleImputer(strategy="median")` learns a median from training data and fills missing values.
2. `StandardScaler()` centers and scales numeric variables. This is useful for many linear and distance-based models.

### Categorical variables

1. Missing categories are replaced with the most frequent training category.
2. One-hot encoding creates indicator columns.
3. Infrequent categories are grouped based only on the data used during `fit`.
4. A previously unseen category can be processed safely during inference.

In [7]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=0.01,
                drop="if_binary",
            ),
        ),
    ]
)

## 8. Combine the transformations with `ColumnTransformer`

`remainder="drop"` means that only the explicitly listed features reach the model. This makes feature selection visible and auditable.

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
        ("binary", "passthrough", binary_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

## 9. Put feature engineering inside the preparation pipeline

Although we previewed the engineered features earlier, the real workflow must recreate them inside the pipeline. This ensures that the same code is applied during training, validation, test evaluation, and future inference.

`fit_transform` below is used only as a structural demonstration on training data. During model comparison, the complete pipeline must be fitted separately inside every cross-validation fold.

In [9]:
feature_engineering = FunctionTransformer(
    create_titanic_features,
    validate=False,
)

preparation_pipeline = Pipeline(
    steps=[
        ("feature_engineering", feature_engineering),
        ("preprocessor", preprocessor),
    ]
)

X_train_prepared = preparation_pipeline.fit_transform(
    X_train,
    y_train,
)

output_feature_names = (
    preparation_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print(f"Raw training shape: {X_train.shape}")
print(f"Prepared training shape: {X_train_prepared.shape}")

pd.DataFrame({"output_feature": output_feature_names})

Raw training shape: (712, 11)
Prepared training shape: (712, 18)


,output_feature
0,Age
1,FareLog1p
2,FamilySize
3,Sex_male
4,Embarked_C
5,Embarked_Q
6,Embarked_S
7,Pclass_1
8,Pclass_2
9,Pclass_3


### Basic structural checks

These checks verify that:

- the number of passengers did not change;
- every output value is finite.

In [10]:
assert X_train_prepared.shape[0] == X_train.shape[0]
assert X_train_prepared.shape[1] == len(output_feature_names)

if hasattr(X_train_prepared, "toarray"):
    values_to_check = X_train_prepared.toarray()
else:
    values_to_check = X_train_prepared

assert np.isfinite(values_to_check).all()

print("All structural checks passed.")

All structural checks passed.


## 10. Build the future raw-data-to-model pipeline

The modeling notebook will supply an estimator. `clone` creates an unfitted copy of the preprocessor so that different experiments do not share learned state.

The model will therefore receive raw Titanic rows; feature engineering, imputation, encoding, scaling, and prediction will remain together.

In [11]:
def build_model_pipeline(estimator) -> Pipeline:
    """Create an unfitted end-to-end pipeline for a given estimator."""
    return Pipeline(
        steps=[
            (
                "feature_engineering",
                FunctionTransformer(
                    create_titanic_features,
                    validate=False,
                ),
            ),
            ("preprocessor", clone(preprocessor)),
            ("model", estimator),
        ]
    )

# Example for the modeling notebook:
# from sklearn.linear_model import LogisticRegression
# model_pipeline = build_model_pipeline(
#     LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)
# )

## 11. Save a readable split manifest

We save passenger-to-partition assignments so that another notebook can reproduce the exact split. The construction is intentionally explicit: create each table, label it, combine both tables, and sort the result.

We do **not** save the already transformed matrix. The final model pipeline must learn its preprocessing inside cross-validation.

In [12]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

train_manifest = X_train[["PassengerId"]].copy()
train_manifest["split"] = "train"

test_manifest = X_test[["PassengerId"]].copy()
test_manifest["split"] = "test"

split_manifest = pd.concat(
    [train_manifest, test_manifest],
    ignore_index=True,
)

split_manifest = split_manifest.sort_values("PassengerId")

split_manifest.to_csv(
    ARTIFACTS_DIR / "split_assignments_teaching.csv",
    index=False,
)

split_manifest.head()

,PassengerId,split
607,1,train
209,2,train
142,3,train
633,4,train
810,5,test


## 12. Save minimal reproducibility metadata

At this stage there is no selected fitted model, so we do not serialize a production pipeline. We save only the information needed to reproduce the split and environment.

In [13]:
metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_column": TARGET_COLUMN,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "test_note": (
        "The previous EDA inspected the complete labeled dataset. "
        "Treat this as a test set, not a perfectly untouched final test."
    ),
}

metadata_path = ARTIFACTS_DIR / "data_preparation_teaching_metadata.json"

with metadata_path.open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

print(f"Metadata saved to: {metadata_path}")

Metadata saved to: /mnt/c/Users/aespinoza/PycharmProjects/titanic_ml_solution/artifacts/data_preparation_teaching_metadata.json


## 13. Handoff to modeling

The modeling notebook should:

1. use only `X_train` and `y_train` for cross-validation and model selection;
2. compare complete pipelines created with `build_model_pipeline`;
3. compare the baseline decisions with alternatives from the EDA;
4. select metrics before inspecting model results;
5. evaluate the test set only after selecting the complete pipeline.

### Suggested experiments

- original `Fare` versus `FareLog1p`;
- one-hot versus ordinal `Pclass`;
- raw `SibSp`/`Parch` versus `FamilySize`/`IsAlone`;
- excluding cabin information versus `HasCabin` versus grouped deck;
- median age imputation versus conditional imputation;
- excluding title versus grouped title;
- adding an explicit `Sex × Pclass` interaction for linear models;
- excluding ticket versus a fold-safe ticket-group-size feature.

The advanced companion notebook demonstrates how to implement these alternatives as reusable, configurable transformers.

### Check your understanding

- Why must the imputer learn its median from training data only?
- Why is `PassengerId` excluded from the model?
- Why do we not keep both raw and engineered family variables in the baseline?
- What does `handle_unknown` protect us from during inference?
- Why should the test set not be used during Optuna tuning?